In [ ]:
import cv2
import numpy as np


def is_stain(image: np.ndarray, dark: float = 0.08, min_red: float = 1.8,
             min_area: float = 0.01) -> tuple[bool, float]:
    """A stain = a large patch of paper that is darker AND redder (browner) than the page.
    Shadows and dark edges are darker but not redder, so they are ignored."""
    # Shrink for speed
    s = min(1.0, 300 / max(image.shape[:2]))
    small = cv2.resize(image, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    w = small.shape[1]

    # Paper background: dilate erases ink / bleed-through, median blur smooths
    k = max(3, int(w * 0.006) | 1)
    window = max(3, int(w * 0.04) | 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    bg = cv2.medianBlur(cv2.dilate(small, kernel), window)

    # Darker and redder than the typical paper
    lab = cv2.cvtColor(bg, cv2.COLOR_BGR2LAB).astype(np.float32)
    L, a = lab[:, :, 0], lab[:, :, 1]
    mask = (L < np.median(L) * (1 - dark)) & (a - np.median(a) > min_red)
    mask = cv2.morphologyEx(mask.astype(np.uint8) * 255, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    # Keep only large blobs
    _, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    areas = stats[1:, cv2.CC_STAT_AREA]
    stain_fraction = areas[areas >= 0.005 * mask.size].sum() / mask.size

    return stain_fraction > min_area, float(stain_fraction)


# ---------- Run on Zoernikav folio212 ----------
# img = cv2.imread("StainImages/notstainedimageZoernikav.jpeg")
img = cv2.imread("StainImages/Zoernikav folio212.jpg")

# img = cv2.imread("StainImages/H05.png")

stained, fraction = is_stain(img)
print(f"stain={stained}  stained area={fraction:.4f}")


stain=True  stained area=0.2311


: 